In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, confusion_matrix, classification_report, accuracy_score, roc_auc_score
from sklearn.model_selection import learning_curve, LeaveOneOut, GridSearchCV, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, AdaBoostClassifier
from sklearn.pipeline import make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif, RFE, RFECV, SelectFromModel, SelectFpr, VarianceThreshold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import FunctionTransformer
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC, LinearSVC
from sklearn.neighbors import KNeighborsClassifier

In [20]:
dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_ML_paul.xlsx"
dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_ML_saqib.xlsx"

#dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_paul.xlsx"
#dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/Metabolome_PCA_saqib.xlsx"

#dataset_total_path = "C:/Users/tamer/Documents/PhD/ML/old_metabolome_paul.xlsx"
#dataset_val_path = "C:/Users/tamer/Documents/PhD/ML/old_metabolome_saqib.xlsx"

df = pd.read_excel(dataset_total_path)
#df = df.iloc[:, :sel]
df_val = pd.read_excel(dataset_val_path)
#df_val = df_val.iloc[:, :sel]


In [21]:
print(df.shape)
print(df_val.shape)

(24, 263)
(12, 263)


In [22]:
Class_A = 'LN'
Class_B = 'LP'

df = df[(df["Class"] == Class_A) | (df["Class"] == Class_B)]
df_val = df_val[(df_val["Class"] == Class_A) | (df_val["Class"] == Class_B)]

In [23]:
def encodage(df):
    code = {
    'LP' : 1,
    'SP' : 2,
    'LN' : 3,
    'SN' : 4
}
# Appliquer ce dictionnaire aux colonnes du dataset, avec la fonction map
    for col in df.select_dtypes('object'):
        df[col] = df[col].map(code)

    return df


def preprocessing(df):
    df = encodage(df)

    X = df.drop(['Class'], axis = 1)
    y = df['Class']

    # compter le nombre d'échantillons restants dans le dataset après avoir été inputé
    print(y.value_counts())

    return X, y

In [24]:
X_total, y_total = preprocessing(df)
X_val, y_val = preprocessing(df_val)

Class
3    6
1    6
Name: count, dtype: int64
Class
3    3
1    3
Name: count, dtype: int64


# Preliminary model analysis

In [7]:
def evaluation(model):
    model.fit(X_total, y_total)
    ypred = model.predict(X_val)
    cm = confusion_matrix(y_val, ypred)
    report = classification_report(y_val, ypred)
    
    print(confusion_matrix(y_val, ypred))
    print(classification_report(y_val, ypred))

    # evaluation du modèle en fonction du nombre de samples, basé sur le f1 score
    # Evalue si underfit ou overfit
    N, train_score, val_score = learning_curve(model, X_total, y_total, cv=4, scoring='f1', train_sizes=np.linspace(0.1, 1, 10))

    plt.figure(figsize=(10,6))
    # on prend les .means de chaque round de cv. cv = 4 <=> on sépare le dataset en 4 et prend 3 part = train 1 part = test, puis on échange
    plt.plot(N, train_score.mean(axis=1), label = 'train score')
    plt.plot(N, val_score.mean(axis=1), label = 'val score')
    plt.legend()

    path = "C:/Users/tamer/Documents/PhD/ML/metrics.txt"
    with open(path, 'a', encoding='utf-8') as f:
        f.write(f"\n==== {model} ====\n")
        f.write("Confusion matrix:\n")
        f.write(str(cm) + "\n")
        f.write("Classification report:\n")
        f.write(report + "\n")
        f.write("="*40 + "\n")

In [25]:
X_total.var(axis=0)

Pheophorbide a                                    1.130750e+19
Nobiletin                                         5.295292e+12
Sinensetin                                        3.444500e+12
Adenosine                                         1.915658e+17
D-(-)-Quinic acid                                 5.721040e+17
                                                      ...     
Shanzhiside                                       1.702637e+13
Okanin 4'-(4''-acetyl-6''-p-coumarylglucoside)    2.678198e+13
N-Acetyl-L-rhamnosamine                           3.538091e+12
Artemetin                                         1.055388e+11
Isorhamnetin                                      1.110761e+12
Length: 262, dtype: float64

In [26]:
##########################

log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
X_var2 = log2_transformer.fit_transform(X_total)
X_var2.var(axis=0).median()


0.15510781878424867

In [27]:
##########################

from sklearn.feature_selection import VarianceThreshold

vt = VarianceThreshold(threshold=0.177)
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
preprocessor = make_pipeline(log2_transformer, vt)
X_var = preprocessor.fit_transform(X_total)
X_var.shape



(12, 113)

In [11]:
log2_transformer = FunctionTransformer(lambda x: np.log2(x + 1))
#selector = SelectFpr(f_classif, alpha=0.0005)
selector = VarianceThreshold(threshold=1e-1)
#selector = SelectKBest(score_func = f_classif, k=50)
preprocessor = make_pipeline(log2_transformer, selector)

In [12]:
# ALL the models to evaluate

RandomForest = make_pipeline(preprocessor, RandomForestClassifier(random_state = 0))
AdaBoost = make_pipeline(preprocessor, AdaBoostClassifier(random_state = 0))
GB = make_pipeline(preprocessor, GradientBoostingClassifier(random_state = 0))

SVM = make_pipeline(preprocessor, StandardScaler(), SVC(kernel = 'rbf', random_state = 0))
SVM_linear_L2 = make_pipeline(preprocessor, StandardScaler(), LinearSVC(penalty = 'l2', random_state = 0))
SVM_linear_L1 = make_pipeline(preprocessor, StandardScaler(), LinearSVC(penalty = 'l1', random_state = 0))
KNN = make_pipeline(preprocessor, StandardScaler(), KNeighborsClassifier())
LR_L1 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l1', random_state = 0, solver = 'liblinear'))
LR_L2 = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='l2', random_state = 0, solver = 'liblinear'))
LR_EN = make_pipeline(preprocessor, StandardScaler(), LogisticRegression(penalty='elasticnet', l1_ratio=0.5, random_state = 0, solver = 'saga'))


dict_of_models = {'RandomForest' : RandomForest, 
                  'AdaBoost' : AdaBoost, 
                  'GB' : GB,
                  'SVM' : SVM, 
                  'SVM_linear_L2' : SVM_linear_L2,
                  'SVM_linear_L1' : SVM_linear_L1,
                  'KNN' : KNN,
                  'LR_L1' : LR_L1,
                  'LR_L2' : LR_L2,
                  'LR_EN' : LR_EN
                 }

In [13]:
#cv = LeaveOneOut()

#for name, model in dict_of_models.items():
    #score = cross_val_score(model, X_total, y_total, cv = cv, scoring = 'accuracy').mean()
    #print(name)
    #print(score)

In [ ]:
for name, model in dict_of_models.items():
    print(name)
    evaluation(model)

# Model optimization

In [15]:
SVM_linear_L2.fit(X_total, y_total)
LR_L2.fit(X_total, y_total)

Pipeline(steps=[('pipeline',
                 Pipeline(steps=[('functiontransformer',
                                  FunctionTransformer(func=<function <lambda> at 0x0000026E23744CC0>)),
                                 ('variancethreshold',
                                  VarianceThreshold(threshold=0.1))])),
                ('standardscaler', StandardScaler()),
                ('logisticregression',
                 LogisticRegression(random_state=0, solver='liblinear'))])

In [18]:
mask = SVM_linear_L2.named_steps['pipeline'].named_steps['variancethreshold'].get_support()
feat_names = X_total.columns[mask]
svc = SVM_linear_L2.named_steps['linearsvc']


coef = svc.coef_.ravel()  # shape: (n_selected_features,)
coef_svc_df = pd.DataFrame({'Feature': feat_names, 'Coefficient': coef}) \
            .sort_values('Coefficient', ascending=False)

coef_svc_df.to_excel("C:/Users/tamer/Documents/PhD/ML/coefs_svc.xlsx")

In [19]:
mask = LR_L2.named_steps['pipeline'].named_steps['variancethreshold'].get_support()
feat_names = X_total.columns[mask]
lr = LR_L2.named_steps['logisticregression']


coef = lr.coef_.ravel()  # shape: (n_selected_features,)
coef_lr_df = pd.DataFrame({'Feature': feat_names, 'Coefficient': coef}) \
            .sort_values('Coefficient', ascending=False)

coef_lr_df.to_excel("C:/Users/tamer/Documents/PhD/ML/coefs_lr.xlsx")